***PREPROCESSING — STEP 1: Caricamento dataset***

In [1]:
# Import librerie
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Carico il CSV in un DataFrame
df = pd.read_csv("C:/Users/matteo.segatto/Desktop/TicketClassifier/data/TicketEstrazione270326.csv")

***PREPROCESSING — STEP 2: Normalizzazione campo area***

In [2]:
# Mapping completo chiave → valore dei campi Area numerici, basato sulla tabella zhc_dropdown_maia
AREA_MAPPING = {
    '1': 'Amministrazione',
    '2': 'Amm-Economato',
    '3': 'Utenti e Ospiti',
    '4': 'CSS',
    '5': 'Personale',
    '6': 'Hardware',
    '7': 'CCE',
    '8': 'CBA DR STP',
}

# Quanti ticket hanno area numerica?
area_numerica_mask = df['area'].astype(str).str.strip().isin(AREA_MAPPING.keys())
print(f"Ticket con area numerica da mappare: {area_numerica_mask.sum():,}")
print(df[area_numerica_mask]['area'].value_counts())

# Applico mapping
df['area'] = df['area'].apply(
    lambda x: AREA_MAPPING.get(str(x).strip(), x) if pd.notna(x) else x
)

print(f"\nDistribuzione area dopo mapping:")
print(df['area'].value_counts(dropna=False))

Ticket con area numerica da mappare: 868
area
6    856
1      4
5      4
4      3
3      1
Name: count, dtype: int64

Distribuzione area dopo mapping:
area
area_personale                     14924
ciclo_passivo                      11535
ciclo_attivo                        9583
area_sanitaria                      9100
NaN                                 6403
rendicontazione_flussi              2927
protocollo_documentale_delibere     2716
area_sistemistica                   1173
sistema381                          1055
Hardware                             856
area_territoriale                    555
protocollo_delibere                  228
business_intelligence                159
Amministrazione                        4
Personale                              4
CSS                                    3
Utenti e Ospiti                        1
flussi_regionali                       1
Name: count, dtype: int64


***PREPROCESSING — STEP 3: Imputazione priorita_iniziale_cliente***

In [3]:
# Se trovo NaN in priorità iniziale = operatore NON ha modificato la priorità
# Il cliente aveva già scelto quella giusta
# priorita_iniziale_cliente == priorita_finale
# Imputo i NaN con priorita_finale

nan_prima = df['priorita_iniziale_cliente'].isna().sum()
print(f"NaN prima dell'imputazione: {nan_prima:,} ({nan_prima/len(df)*100:.1f}%)")

df['priorita_iniziale_cliente'] = df['priorita_iniziale_cliente'].fillna(df['priorita_finale'])

nan_dopo = df['priorita_iniziale_cliente'].isna().sum()
print(f"NaN dopo l'imputazione: {nan_dopo:,}")

# Distribuzione dopo imputazione
print(f"\nDistribuzione priorita_iniziale_cliente dopo imputazione:")
print(df['priorita_iniziale_cliente'].value_counts().sort_index())

NaN prima dell'imputazione: 53,857 (88.0%)
NaN dopo l'imputazione: 0

Distribuzione priorita_iniziale_cliente dopo imputazione:
priorita_iniziale_cliente
P1    13373
P2    15572
P3    31040
P4     1242
Name: count, dtype: int64


***PREPROCESSING — STEP 4: Pulizia testi***

In [4]:
# import librerie per pulizia testo
from bs4 import BeautifulSoup
import re

# ─────────────────────────────────────────────────────────────────────────────
# PATTERN DISCLAIMER ZHC
#
# Molti ticket (soprattutto area_sistemistica) contengono in coda la firma
# completa dell'operatore ZHC con indirizzo fisico e il disclaimer GDPR:
#
#   Zucchetti Healthcare Srl       ← inizio firma con indirizzo
#   V.le Trento, 56 | 38068 Rovereto (TN)
#   Il contenuto di questa e-mail e degli eventuali allegati è strettamente
#   confidenziale ...               ← disclaimer GDPR
#   ... ufficio.privacy@zucchetti.it ← fine disclaimer
#
# Questi testi non contengono informazioni utili sulla natura del ticket
# e creano artefatti nel modello (i termini del disclaimer diventano
# falsamente discriminanti per area_sistemistica).
#
# Strategia: tagliamo a partire dal primo anchor riconoscibile.
#   1. "Zucchetti Healthcare Srl"  → rimuove firma + indirizzo + disclaimer
#   2. "Il contenuto di questa e-mail" → fallback solo per il disclaimer GDPR
# ─────────────────────────────────────────────────────────────────────────────

# Anchor 1: nome azienda nella firma — appare quasi esclusivamente nel footer
PATTERN_FIRMA_ZHC = re.compile(
    r'\s*Zucchetti Healthcare Srl.*',
    flags=re.IGNORECASE | re.DOTALL
)

# Anchor 2: inizio del testo del disclaimer GDPR (fallback)
PATTERN_DISCLAIMER_ZHC = re.compile(
    r'\s*Il contenuto di questa e-mail.*',
    flags=re.IGNORECASE | re.DOTALL
)


def rimuovi_disclaimer_zhc(testo: str) -> str:
    """
    Rimuove la firma e il disclaimer GDPR di Zucchetti Healthcare dal testo.

    Prova prima l'anchor più specifico (firma completa con indirizzo),
    poi il solo disclaimer GDPR come fallback.
    Restituisce il testo pulito con spazi extra rimossi.
    """
    if not isinstance(testo, str):
        return testo

    # Prova 1: taglia dalla firma completa (cattura anche indirizzo)
    testo = PATTERN_FIRMA_ZHC.sub('', testo)

    # Prova 2: rimuovi il disclaimer GDPR se ancora presente
    testo = PATTERN_DISCLAIMER_ZHC.sub('', testo)

    return testo.strip()


# ─────────────────────────────────────────────────────────────────────────────
# FUNZIONI DI PULIZIA HTML
# ─────────────────────────────────────────────────────────────────────────────

def pulisci_html(testo):
    """Pulizia base HTML usando BeautifulSoup."""
    # se il valore non è stringa o è vuoto/non significativo restituisco stringa vuota
    if not isinstance(testo, str) or not testo.strip():
        return ""
    # costruisco un DOM con BeautifulSoup
    soup = BeautifulSoup(testo, "html.parser")
    # ogni tag <br> viene sostituito con un newline
    for br in soup.find_all("br"):
        br.replace_with("\n")
    # per ogni <p> inserisco uno spazio dopo il tag e lo "sciolgo"
    # (unwrap rimuove il tag lasciando il contenuto)
    for p in soup.find_all("p"):
        p.insert_after(" ")
        p.unwrap()
    # estraggo il testo finale e tolgo spazi iniziali/finali
    return soup.get_text().strip()


def pulisci_blocco_reperibilita(soup):
    """
    Rimuove il blocco 'Orario reperibilità + Telefono' 
    navigando il DOM.
    """
    # prendo tutti i <br> perchè da lì riconosco la struttura del blocco
    br_tags = soup.find_all("br")
    testo = soup.get_text()
    
    # se nel testo compare la stringa di reperibilità (con o senza accento)
    if "Orario reperibilità" in testo or "Orario reperibilita" in testo:
        # verifico che ci siano almeno due <br> (atteso formattazione del blocco)
        if len(br_tags) >= 2:
            second_br = br_tags[1]
            # estraggo tutti i nodi precedenti al secondo <br> (cioè l'intero blocco)
            for elem in list(second_br.previous_siblings):
                elem.extract()
            # estraggo anche il secondo <br> stesso
            second_br.extract()
    return soup


def pulisci_descrizione(testo):
    """
    Pulizia descrizione ticket:
    - Rimuove blocco reperibilità/telefono
    - Rimuove HTML
    - Rimuove firma e disclaimer GDPR ZHC
    """
    # guard clause: input non valido → stringa vuota
    if not isinstance(testo, str) or not testo.strip():
        return ""
    
    # parse HTML
    soup = BeautifulSoup(testo, "html.parser")
    # elimino l'eventuale blocco di reperibilità
    soup = pulisci_blocco_reperibilita(soup)
    
    # normalizzo i tag <br> e <p> come in pulisci_html
    for br in soup.find_all("br"):
        br.replace_with("\n")
    for p in soup.find_all("p"):
        p.insert_after(" ")
        p.unwrap()
    
    testo_pulito = soup.get_text()
    # tolgo triple (o più) newline consecutivi -> lascio al massimo due
    testo_pulito = re.sub(r'\n{3,}', '\n\n', testo_pulito)

    # rimuovo firma e disclaimer ZHC (artefatto che inquina il testo)
    testo_pulito = rimuovi_disclaimer_zhc(testo_pulito)

    return testo_pulito.strip()


def pulisci_conversazione(testo):
    """
    Pulizia conversazione completa:
    - Divide per separatore ---
    - Pulisce ogni messaggio singolarmente
    - Rimuove blocco reperibilità solo dal primo messaggio cliente
    - Ricostruisce la conversazione
    """
    # guard clause
    if not isinstance(testo, str) or not testo.strip():
        return ""
    
    # i messaggi sono separati dal marcatore esplicito
    messaggi = testo.split('\n---\n')
    messaggi_puliti = []
    
    # itero su tutti i messaggi
    for i, messaggio in enumerate(messaggi):
        # cerco di separare l'autore dal corpo (formato 'AUTORE: testo')
        if ': ' in messaggio:
            autore, corpo = messaggio.split(': ', 1)
        else:
            autore, corpo = '', messaggio
        
        soup = BeautifulSoup(corpo, "html.parser")
        
        # solo se l'autore è CLIENTE rimuovo il blocco reperibilità
        if autore.upper() == 'CLIENTE':
            soup = pulisci_blocco_reperibilita(soup)
        
        # sostituisco <br> e <p> come prima
        for br in soup.find_all("br"):
            br.replace_with("\n")
        for p in soup.find_all("p"):
            p.insert_after(" ")
            p.unwrap()
        
        corpo_pulito = soup.get_text().strip()
        
        # se il messaggio è vuoto dopo la pulizia lo salto
        if not corpo_pulito:
            continue
        
        # ricostruisco la stringa del messaggio con autore se presente
        if autore:
            if autore.upper() == 'CLIENTE':
                autore_norm = 'CLIENTE'
            else:
                autore_norm = 'OPERATORE'
            messaggi_puliti.append(f"{autore_norm}: {corpo_pulito}")
        else:
            messaggi_puliti.append(corpo_pulito)
    
    # riunisco la conversazione con lo stesso separatore iniziale
    return '\n---\n'.join(messaggi_puliti)

# Applica le funzioni
df['oggetto_pulito']        = df['oggetto'].apply(pulisci_html)
df['descrizione_pulita']    = df['descrizione'].apply(pulisci_descrizione)
df['conversazione_pulita']  = df['conversazione'].apply(pulisci_conversazione)

In [5]:
# Analisi lunghezza descrizione
df['desc_n_char'] = df['descrizione_pulita'].str.len()
df['desc_n_parole'] = df['descrizione_pulita'].str.split().str.len()

print("=== STATISTICHE LUNGHEZZA DESCRIZIONE ===")
print(df[['desc_n_char', 'desc_n_parole']].describe().round(1))
print(f"\nDescrizioni vuote o quasi (< 5 parole): {(df['desc_n_parole'] < 5).sum():,}")
print(f"Descrizioni molto corte (< 10 parole):   {(df['desc_n_parole'] < 10).sum():,}")

=== STATISTICHE LUNGHEZZA DESCRIZIONE ===
       desc_n_char  desc_n_parole
count      61227.0        61227.0
mean         363.2           52.1
std          601.1           74.1
min            0.0            0.0
25%          184.0           26.0
50%          283.0           41.0
75%          421.0           62.0
max        65533.0         8673.0

Descrizioni vuote o quasi (< 5 parole): 794
Descrizioni molto corte (< 10 parole):   2,579


***Test***

***Test rimozione disclaimer ZHC***

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# TEST RIMOZIONE DISCLAIMER ZHC
#
# Verifichiamo tre cose:
#   1. Il disclaimer viene rimosso correttamente nei ticket che ce l'hanno
#   2. I ticket senza disclaimer non vengono modificati (test di regressione)
#   3. Nessun ticket diventa vuoto per effetto della rimozione
# ─────────────────────────────────────────────────────────────────────────────

ANCHOR_DISCLAIMER = 'il contenuto di questa e-mail'

# Quante descrizioni ancora contengono il disclaimer dopo la pulizia?
n_residui = df['descrizione_pulita'].str.lower().str.contains(ANCHOR_DISCLAIMER, na=False).sum()
print(f'Ticket con disclaimer ancora presente dopo pulizia: {n_residui}')
print(f'(atteso: 0)\n')

# Quante descrizioni sono diventate vuote dopo la pulizia?
n_vuote = (df['descrizione_pulita'].str.strip() == '').sum()
print(f'Descrizioni vuote dopo pulizia: {n_vuote}')
print(f'(dovrebbe essere uguale o vicino al valore precedente alla modifica)\n')

# Mostra un esempio prima/dopo per verifica visiva
# Cerchiamo un ticket che aveva il disclaimer nel grezzo ma non nella descrizione pulita
mask_aveva_disclaimer = df['descrizione'].str.contains('Zucchetti Healthcare Srl', case=False, na=False)
print(f'Ticket con firma ZHC nella descrizione grezza: {mask_aveva_disclaimer.sum():,}')

if mask_aveva_disclaimer.sum() > 0:
    idx_esempio = df[mask_aveva_disclaimer].index[0]
    print(f'\n--- DESCRIZIONE GREZZA (ultimi 400 caratteri) ---')
    print(df.loc[idx_esempio, 'descrizione'][-400:])
    print(f'\n--- DESCRIZIONE PULITA (ultimi 400 caratteri) ---')
    print(df.loc[idx_esempio, 'descrizione_pulita'][-400:])

Ticket con disclaimer ancora presente dopo pulizia: 0
(atteso: 0)

Descrizioni vuote dopo pulizia: 117
(dovrebbe essere uguale o vicino al valore precedente alla modifica)

Ticket con firma ZHC nella descrizione grezza: 2,941

--- DESCRIZIONE GREZZA (ultimi 400 caratteri) ---
tuali costi?
Ringrazio per l’attenzione e resto a disposizione per eventuali delucidazioni.
Cordiali saluti.

Paola Gianoli – Responsabile Ufficio Amministrazione Risorse Umane
Fondazione Mons. Gerolamo Comi – ONLUS
Via Forlanini 6 - 21016 Luino (VA)
Tel. 0332.542335
Sito web: www.fondazionecomi.it
E-mail: p.gianoli@fondazionecomi.it
Pec: fondazionecomi@pec.it






Privo di virus.www.avg.com






--- DESCRIZIONE PULITA (ultimi 400 caratteri) ---
ciao ragazze,
in allegato l'offerta per attivare la scrittura degli Stipendi da CBA DR verso l'Economica 2.0 di SIPCAR.
Grazie.

Massimo PasqualiniCommerciale


In [7]:
mask_rep = df['conversazione'].str.contains('reperibilit', case=False, na=False)
print(f"Ticket con blocco reperibilità nella conversazione grezza: {mask_rep.sum():,}")

if mask_rep.sum() > 0:
    idx = df[mask_rep].index[4]
    print("\n=== CONVERSAZIONE GREZZA (primi 6000 chars) ===")
    print(df.loc[idx, 'conversazione'][:6000])
    print("\n=== CONVERSAZIONE PULITA (primi 6000 chars) ===")
    print(df.loc[idx, 'conversazione_pulita'][:6000])

Ticket con blocco reperibilità nella conversazione grezza: 27,019

=== CONVERSAZIONE GREZZA (primi 6000 chars) ===
CLIENTE: <strong>Orario reperibilità contatto: </strong><br /><strong>Telefono: </strong><br /><div class="ql-editor"><p>CIAO IVAN,</p><p><br /></p><p>Stiamo cambiando connettività nelle varie strutture adesso presso la Casa di Riposo Villa del Pensionato - Via San Francesco 3 - 47017 Rocca San Casciano (FC) l'indirizzo IP pubblico è il seguente: 79.62.249.228</p><p><br /></p><p> </p><p><br /></p><p>Si segnala l'urgenza</p><p><br /></p><p> </p><p><br /></p><p>Cordiali Saluti</p></div>
---
Mauro Zaltron: <p>Effettuata modifica IP</p><br />
<p>attendo riscontro</p>
---
CLIENTE: <div class="ql-editor"><p>LA STRUTTURA MI HA COMUNICATO CHE FUNZIONANO I PROGRAMMI 1.0 CHE SI UTILLIZZANO TRAMITE DESKTOP REMOTO</p></div>
---
Mauro Zaltron: <div class="mozaik-inner" style="font-family:Arial, Helvetica, sans-serif;font-size:14px;line-height:22.4px;color:#444444;background-color:rgba(

In [8]:
# Visualizzo l'url del ticket per verifica manuale
print(f"\nURL del ticket: {df.loc[idx, 'url_ticket']}")


URL del ticket: https://maia.zucchettihc.it/index.php?module=Cases&action=DetailView&record=10160380-f462-cbd5-287a-67584f55e30f


***PREPROCESSING — STEP 6: Costruzione testo_input (oggetto_pulito + descrizione_pulita)***

In [9]:

# STEP 6: Costruzione testo_input
# Concatenazione oggetto_pulito + spazio + descrizione_pulita
# fillna('') per gestire i NaN, strip() per rimuovere spazi iniziali/finali
df['testo_input'] = (df['oggetto_pulito'].fillna('') + ' ' + df['descrizione_pulita'].fillna('')).str.strip()

# rimozione boilerplate residuo da oggetto — applicata direttamente sul testo finale
df['testo_input'] = df['testo_input'].str.replace(
    r'\s*Orario reperibilit\S+.*?(?=\n|Buongiorno|Salve|Ciao|$)',
    '',
    regex=True,
    flags=re.IGNORECASE
).str.strip()

# verifica
n = df['testo_input'].str.contains('Orario reperibilit', case=False, na=False).sum()
print(f"Boilerplate residuo: {n}")

# Statistiche
testi_vuoti = (df['testo_input'] == '').sum()
lunghezza_media = df['testo_input'].str.split().str.len().mean()

print(f"Testi vuoti (stringa vuota dopo strip): {testi_vuoti:,}")
print(f"Lunghezza media in parole:              {lunghezza_media:.1f}")

print(f"\nEsempio testo_input primi 600 caratteri):")
esempio = df[df['testo_input'] != '']['testo_input'].iloc[3]
print(esempio[:600])

# visualizzo url del ticket per verifica manuale
print(f"\nURL del ticket: {df.loc[df['testo_input'] != '', 'url_ticket'].iloc[3]}")


Boilerplate residuo: 0
Testi vuoti (stringa vuota dopo strip): 0
Lunghezza media in parole:              53.5

Esempio testo_input primi 600 caratteri):
Bilancio di previsione Buongiorno,
sto cercando di accedere a "Bilancio di previsione", ma mi compare l'errore in allegato.
Come devo procedere?
Grazie
Laura

URL del ticket: https://maia.zucchettihc.it/index.php?module=Cases&action=DetailView&record=100255c2-d55c-bb9c-8e2e-691da5650cc5


***PREPROCESSING — STEP 8: Rimozione ticket senza informazione utile***

In [10]:
# Mostriamo la distribuzione prima di filtrare
print("=== Distribuzione lunghezze testo_input ===")
bins   = [0, 3, 5, 10, 20, 50, float('inf')]
labels = ['<=3', '3-5', '5-10', '10-20', '20-50', '50+']
df['n_parole'] = df['testo_input'].str.split().str.len()
print(pd.cut(df['n_parole'], bins=bins, labels=labels).value_counts().sort_index())

# Scegliamo di rimuovere i ticket con meno di 3 parole
# Sotto questa soglia non c'è abbastanza testo per capire il problema
mask_troppo_corti = df['n_parole'] < 3

print(f"\nTicket da rimuovere (< 3 parole): {mask_troppo_corti.sum():,}")
print(f"Ticket rimanenti:                  {(~mask_troppo_corti).sum():,}")

# Mostriamo qualche esempio di quello che stiamo rimuovendo
print("\n=== Esempi ticket rimossi ===")
print(df[mask_troppo_corti][['testo_input', 'priorita_finale']].head(10).to_string())

# Rimozione
df = df[~mask_troppo_corti].copy()

=== Distribuzione lunghezze testo_input ===
n_parole
<=3        129
3-5        326
5-10      1364
10-20     6880
20-50    29290
50+      23238
Name: count, dtype: int64

Ticket da rimuovere (< 3 parole): 71
Ticket rimanenti:                  61,156

=== Esempi ticket rimossi ===
                  testo_input priorita_finale
631     aggiornamento\n\nmmmm              P4
716          PARTENZA CLIENTE              P3
4782            squillo\n\nCU              P2
5037       aggiornamento 2025              P3
5743   ordinativo informatico              P2
6426          VERIFICA UTENTI              P3
6495      fattura elettornica              P3
9741      blocco elaborazione              P2
11563                  errori              P3
13523        FORMAZIONE SVAMA              P3


In [11]:
df.columns

Index(['url_ticket', 'case_number', 'oggetto', 'descrizione',
       'priorita_finale', 'priorita_iniziale_cliente', 'area',
       'tipologia_intervento', 'articolo', 'modulo_sw', 'reparto',
       'data_creazione', 'data_risoluzione', 'conversazione', 'oggetto_pulito',
       'descrizione_pulita', 'conversazione_pulita', 'desc_n_char',
       'desc_n_parole', 'testo_input', 'n_parole'],
      dtype='str')

***PREPROCESSING — STEP 9b: Colonne booleane keyword per AreaClassifier***

Feature binarie ricavate da keyword statisticamente discriminanti per le classi
con F1 basso (analisi TF-IDF + chi-quadro su `KeywordAnalysis_Area.ipynb`).

Ogni colonna vale `True` se il termine compare in `testo_input`, `False` altrimenti.
Le colonne vengono poi usate come feature aggiuntive nell'AreaClassifier
(concatenate alla matrice embedding prima del fit LinearSVC).

In [12]:
testo = df['testo_input'].str.lower().fillna('')

# ── sistema381 ────────────────────────────────────────────────────────────────
df['kw_s381_codice'] = (
    testo.str.contains(r'\b381\b', regex=True) |
    testo.str.contains('sistema 381', regex=False) |
    testo.str.contains('sistema381',  regex=False)
)
df['kw_s381_rapportino'] = (
    testo.str.contains('rapportino', regex=False) |
    testo.str.contains('rapportini', regex=False)
)
df['kw_s381_timbrate']            = testo.str.contains('timbrate',           regex=False)
df['kw_s381_calendario_presenze'] = testo.str.contains('calendario presenze', regex=False)

# ── area_territoriale ─────────────────────────────────────────────────────────
df['kw_ter_unodomo'] = (
    testo.str.contains('unodomo',   regex=False) |
    testo.str.contains('uno domo',  regex=False) |
    testo.str.contains(r'\bdomo\b', regex=True)
)
df['kw_ter_distretto'] = testo.str.contains('distretto', regex=False)

# ── area_sanitaria — NUOVE ────────────────────────────────────────────────────
# Terminologia clinica esclusiva del software sanitario.
# Precision 0.86-0.97 dal chi2 — quasi mai in altre aree.
df['kw_san_terapia'] = (
    testo.str.contains('terapia',  regex=False) |
    testo.str.contains('terapie',  regex=False)
)
df['kw_san_pai']         = testo.str.contains('pai',          regex=False)
df['kw_san_css']         = testo.str.contains('css',          regex=False)
df['kw_san_diario']      = testo.str.contains('diario',       regex=False)
df['kw_san_contenzioni'] = testo.str.contains('contenzioni',  regex=False)
df['kw_san_farmaco']     = testo.str.contains('farmaco',      regex=False)

# ── ciclo_passivo ─────────────────────────────────────────────────────────────
df['kw_pas_iva']       = testo.str.contains('iva',       regex=False)
df['kw_pas_cespiti']   = (
    testo.str.contains('cespiti', regex=False) |
    testo.str.contains('cespite', regex=False)
)
df['kw_pas_prima_nota']   = testo.str.contains('prima nota',   regex=False)
df['kw_pas_ammortamento'] = testo.str.contains('ammortamento', regex=False)
df['kw_pas_analitica']    = testo.str.contains('analitica',    regex=False)
df['kw_pas_reverse']      = testo.str.contains('reverse',      regex=False)

# kw_pas_fornitore RIVISTA — richede contesto contabile per evitare
# falsi positivi su ticket ciclo_attivo che menzionano fornitori
df['kw_pas_fornitore'] = (
    (
        testo.str.contains('fornitore', regex=False) |
        testo.str.contains('fornitori', regex=False)
    ) &
    (
        testo.str.contains('registrazione', regex=False) |
        testo.str.contains('pagamento',     regex=False) |
        testo.str.contains('iva',           regex=False) |
        testo.str.contains('fattura',       regex=False)
    )
)

# ── ciclo_attivo ──────────────────────────────────────────────────────────────
df['kw_att_retta'] = (
    testo.str.contains('retta', regex=False) |
    testo.str.contains('rette', regex=False)
)
df['kw_att_pagopa']         = testo.str.contains('pagopa',         regex=False)
df['kw_att_sdd']            = testo.str.contains('sdd',            regex=False)
df['kw_att_portale_utenti'] = testo.str.contains('portale utenti', regex=False)

# kw_att_fattura_elettronica RIVISTA — richiede contesto di invio/emissione
# per non catturare fatture elettroniche passive (ciclo_passivo)
df['kw_att_fattura_elettronica'] = (
    testo.str.contains('fattura elettronica', regex=False) &
    (
        testo.str.contains('sdi',       regex=False) |
        testo.str.contains('invio',     regex=False) |
        testo.str.contains('scarto',    regex=False) |
        testo.str.contains('emissione', regex=False)
    )
)

# ── area_sistemistica — NUOVA ─────────────────────────────────────────────────
# "installazione programma" ha precision 0.795 — più specifica di
# "installazione" da sola (0.381)
df['kw_sis_installazione_programma'] = testo.str.contains(
    'installazione programma', regex=False
)

# ── Verifica distribuzione ────────────────────────────────────────────────────
KW_COLS = [c for c in df.columns if c.startswith('kw_')]
print(f'Colonne keyword: {len(KW_COLS)}')
print()
print(f'{"Colonna":<40} {"True":>8} {"True%":>7}')
print('-' * 58)
for col in KW_COLS:
    n = df[col].sum()
    print(f'{col:<40} {n:>8,} {n/len(df)*100:>6.1f}%')

Colonne keyword: 25

Colonna                                      True   True%
----------------------------------------------------------
kw_s381_codice                                481    0.8%
kw_s381_rapportino                            143    0.2%
kw_s381_timbrate                               41    0.1%
kw_s381_calendario_presenze                    38    0.1%
kw_ter_unodomo                                169    0.3%
kw_ter_distretto                               77    0.1%
kw_san_terapia                              1,273    2.1%
kw_san_pai                                  1,756    2.9%
kw_san_css                                  1,171    1.9%
kw_san_diario                                 386    0.6%
kw_san_contenzioni                            218    0.4%
kw_san_farmaco                                365    0.6%
kw_pas_iva                                 11,571   18.9%
kw_pas_cespiti                                772    1.3%
kw_pas_prima_nota                             532 

In [13]:
# ── sistema381 — NUOVE ────────────────────────────────────────────────────────

# Terminologia operativa del gestionale 381
df['kw_s381_commessa'] = (
    testo.str.contains('commessa', regex=False) |
    testo.str.contains('commesse', regex=False)
)
df['kw_s381_lotto']   = testo.str.contains('lotto',   regex=False)
df['kw_s381_reinvio'] = testo.str.contains('reinvio', regex=False)

# Nomi del referente tecnico sistema381 — precision ~1.0 ma fragili:
# se Luca Noriller cambia ruolo questa feature va rimossa o aggiornata.
# Tenerla finché il modello è in produzione, rivedere ad ogni retrain.
df['kw_s381_noriller'] = (
    testo.str.contains('noriller',      regex=False) |
    testo.str.contains('luca noriller', regex=False) |
    testo.str.contains('ciao luca',     regex=False)
)
df['kw_s381_cescatti'] = testo.str.contains('cescatti', regex=False)

# ── area_territoriale — NUOVE ─────────────────────────────────────────────────

# SAD (Servizio di Assistenza Domiciliare) — recall 0.201, cattura 1 ticket su 5
# Precision bassa (0.399) ma è uno dei pochi termini con recall decente
# per questa classe rarissima. Usata in combinazione con unodomo bilancia il rumore.
df['kw_ter_sad'] = testo.str.contains('sad', regex=False)

# UDO (Unità d'Offerta) — terminologia specifica dei servizi territoriali lombardi
df['kw_ter_udo'] = testo.str.contains(r'\budo\b', regex=True)

# Giada — software specifico usato in area territoriale
df['kw_ter_giada'] = testo.str.contains('giada', regex=False)

# ── area_sistemistica — NUOVE ─────────────────────────────────────────────────

# Sipcar — gestionale interno ZHC per la parte sistemistica
df['kw_sis_sipcar'] = testo.str.contains('sipcar', regex=False)

# Termini di accesso remoto — precision alta, inequivocabili
df['kw_sis_vpn'] = testo.str.contains('vpn', regex=False)
df['kw_sis_rdp'] = (
    testo.str.contains(r'\brdp\b',    regex=True) |
    testo.str.contains('desktop remoto', regex=False)
)

# Cloud/server — precision moderata ma recall utile per questa classe
# "cloud" da solo cattura anche ciclo_attivo (cloud fatturazione) quindi
# richiedemento di contesto infrastrutturale
df['kw_sis_server_cloud'] = (
    (
        testo.str.contains('server',  regex=False) |
        testo.str.contains('cloud',   regex=False) |
        testo.str.contains(r'\bvm\b', regex=True)
    ) &
    (
        testo.str.contains('installazione', regex=False) |
        testo.str.contains('accesso',       regex=False) |
        testo.str.contains('configurazione',regex=False) |
        testo.str.contains('aruba',         regex=False)
    )
)

# ── ciclo_passivo — NUOVE ─────────────────────────────────────────────────────

# Bilancio — precision 0.829, termine quasi esclusivo del ciclo passivo
df['kw_pas_bilancio'] = testo.str.contains('bilancio', regex=False)

# Registrazione contabile — precision 0.781
# "registrazione" da sola cattura anche area_sanitaria (registrazione paziente)
# quindi richiede contesto contabile
df['kw_pas_registrazione'] = (
    testo.str.contains('registrazione', regex=False) &
    (
        testo.str.contains('fattura',   regex=False) |
        testo.str.contains('contabile', regex=False) |
        testo.str.contains('iva',       regex=False) |
        testo.str.contains('nota',      regex=False)
    )
)

# ── ciclo_attivo — NUOVE ──────────────────────────────────────────────────────

# Fatturazione attiva — precision 0.739, complementa kw_att_fattura_elettronica
# che richiede contesto SDI/invio. Qui catturiamo anche fatturazione generica
df['kw_att_fatturazione'] = (
    testo.str.contains('fatturazione', regex=False) &
    ~(  # escludiamo contesto passivo per non rubare ticket a ciclo_passivo
        testo.str.contains('fornitore', regex=False) |
        testo.str.contains('cespiti',   regex=False) |
        testo.str.contains('prima nota',regex=False)
    )
)

# SDI senza obbligo di "fattura elettronica" — cattura scarti e anomalie invio
df['kw_att_sdi'] = (
    testo.str.contains(r'\bsdi\b', regex=True) &
    ~testo.str.contains('fornitore', regex=False)
)

# ── Verifica distribuzione aggiornata ─────────────────────────────────────────
KW_COLS = [c for c in df.columns if c.startswith('kw_')]
print(f'Colonne keyword totali: {len(KW_COLS)}')
print()
print(f'{"Colonna":<45} {"True":>8} {"True%":>7}')
print('-' * 63)
for col in KW_COLS:
    n = df[col].sum()
    print(f'{col:<45} {n:>8,} {n/len(df)*100:>6.1f}%')

Colonne keyword totali: 41

Colonna                                           True   True%
---------------------------------------------------------------
kw_s381_codice                                     481    0.8%
kw_s381_rapportino                                 143    0.2%
kw_s381_timbrate                                    41    0.1%
kw_s381_calendario_presenze                         38    0.1%
kw_ter_unodomo                                     169    0.3%
kw_ter_distretto                                    77    0.1%
kw_san_terapia                                   1,273    2.1%
kw_san_pai                                       1,756    2.9%
kw_san_css                                       1,171    1.9%
kw_san_diario                                      386    0.6%
kw_san_contenzioni                                 218    0.4%
kw_san_farmaco                                     365    0.6%
kw_pas_iva                                      11,571   18.9%
kw_pas_cespiti            

***Salvataggio Dataframe Pulito***

In [14]:
# STEP 9 — Selezione colonne finali e salvataggio dataset_clean.csv

KW_COLS = [c for c in df.columns if c.startswith('kw_')]

COLONNE_FINALI = [
    'url_ticket',
    'case_number',
    'testo_input',
    'priorita_finale',
    'priorita_iniziale_cliente',
    'area',
    'articolo',
    'modulo_sw',
    'data_creazione',
] + KW_COLS

df_clean = df[COLONNE_FINALI].copy()

print(f"Righe: {len(df_clean):,}")
print(f"Colonne: {len(df_clean.columns)}  ({len(KW_COLS)} keyword)")
print(f"\nNaN per colonna (non-keyword):")
print(df_clean[COLONNE_FINALI[:11]].isnull().sum())
print(f"\nDistribuzione priorita_finale:")
print(df_clean['priorita_finale'].value_counts().sort_index())

df_clean.to_csv("../data/dataset_clean.csv", index=False, encoding='utf-8-sig')
print("\nSalvato in: data/dataset_clean.csv ✅")

Righe: 61,156
Colonne: 50  (41 keyword)

NaN per colonna (non-keyword):
url_ticket                      0
case_number                     0
testo_input                     0
priorita_finale                 0
priorita_iniziale_cliente       0
area                         6400
articolo                     6432
modulo_sw                    6358
data_creazione                  0
kw_s381_codice                  0
kw_s381_rapportino              0
dtype: int64

Distribuzione priorita_finale:
priorita_finale
P1     8381
P2    17737
P3    33496
P4     1542
Name: count, dtype: int64

Salvato in: data/dataset_clean.csv ✅
